In [ ]:
# PCA + Gaussian Mixture Model Clustering Analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("PCA + GAUSSIAN MIXTURE MODEL CLUSTERING ANALYSIS")
print("=" * 80)

# Load and prepare the data
df = pd.read_csv('data/csv/processed_data.csv')

# Identify feature columns (wavelengths)
non_feature_cols = ['tire_number', 'origin', 'measurement_id']
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
wavelength_columns = [col for col in numeric_columns if col not in non_feature_cols]

# Prepare features and true labels
X = df[wavelength_columns]
y_true = df['origin']  # True component labels
components = y_true.unique()

print(f"Dataset shape: {X.shape}")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of wavelengths: {X.shape[1]}")
print(f"True classes: {components}")
print(f"True number of classes: {len(components)}")

# Step 1: PCA Analysis
print("\n📊 Step 1: PCA Dimensionality Reduction Analysis")
print("-" * 50)

# Find optimal number of PCA components
variance_ratios = []
cumulative_variance = []
n_components_range = range(2, min(101, X.shape[1]))

# Fit PCA with different numbers of components
pca_full = PCA()
pca_full.fit(X)

for n in n_components_range:
    cumulative_var = np.sum(pca_full.explained_variance_ratio_[:n])
    cumulative_variance.append(cumulative_var)

# Find elbow point (95% variance explained)
target_variance = 0.95
optimal_pca_components = None
for i, var in enumerate(cumulative_variance):
    if var >= target_variance:
        optimal_pca_components = n_components_range[i]
        break

if optimal_pca_components is None:
    optimal_pca_components = min(50, X.shape[1] - 1)

print(f"Optimal PCA components for {target_variance*100}% variance: {optimal_pca_components}")
print(f"Variance explained with {optimal_pca_components} components: {cumulative_variance[optimal_pca_components-2]:.3f}")

# Apply PCA with optimal components
pca = PCA(n_components=optimal_pca_components)
X_pca = pca.fit_transform(X)

print(f"Reduced dataset shape: {X_pca.shape}")

# Plot PCA variance analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Cumulative variance plot
ax1.plot(n_components_range, cumulative_variance, 'b-', linewidth=2, marker='o', markersize=4)
ax1.axhline(y=target_variance, color='r', linestyle='--', alpha=0.7, label=f'{target_variance*100}% variance')
ax1.axvline(x=optimal_pca_components, color='r', linestyle='--', alpha=0.7, label=f'Optimal: {optimal_pca_components} components')
ax1.set_xlabel('Number of PCA Components')
ax1.set_ylabel('Cumulative Variance Explained')
ax1.set_title('PCA Cumulative Variance Explained')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Individual variance plot (first 20 components)
individual_variance = pca_full.explained_variance_ratio_[:20]
ax2.bar(range(1, len(individual_variance) + 1), individual_variance, alpha=0.7)
ax2.set_xlabel('PCA Component')
ax2.set_ylabel('Individual Variance Explained')
ax2.set_title('Individual Variance per PCA Component (First 20)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Step 2: Gaussian Mixture Model Clustering
print("\n🎯 Step 2: Gaussian Mixture Model Clustering")
print("-" * 50)

# Test different numbers of clusters
n_clusters_range = range(11, 21)
gmm_results = {
    'n_clusters': [],
    'aic': [],
    'bic': [],
    'log_likelihood': [],
    'silhouette': [],
    'ari': [],  # Adjusted Rand Index
    'nmi': []   # Normalized Mutual Information
}

print("Testing different numbers of clusters...")
print(f"{'N_Clusters':<10} {'AIC':<12} {'BIC':<12} {'Log_Likelihood':<15} {'Silhouette':<12} {'ARI':<8} {'NMI':<8}")
print("-" * 80)

for n_clusters in n_clusters_range:
    # Fit Gaussian Mixture Model
    gmm = GaussianMixture(n_components=n_clusters, random_state=42, covariance_type='full')
    cluster_labels = gmm.fit_predict(X_pca)
    
    # Calculate metrics
    aic = gmm.aic(X_pca)
    bic = gmm.bic(X_pca)
    log_likelihood = gmm.score(X_pca)
    
    # Silhouette score (higher is better)
    if n_clusters > 1:
        silhouette = silhouette_score(X_pca, cluster_labels)
    else:
        silhouette = -1
    
    # Compare with true labels (if available)
    # Convert true labels to numeric
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y_true_numeric = le.fit_transform(y_true)
    
    ari = adjusted_rand_score(y_true_numeric, cluster_labels)
    nmi = normalized_mutual_info_score(y_true_numeric, cluster_labels)
    
    # Store results
    gmm_results['n_clusters'].append(n_clusters)
    gmm_results['aic'].append(aic)
    gmm_results['bic'].append(bic)
    gmm_results['log_likelihood'].append(log_likelihood)
    gmm_results['silhouette'].append(silhouette)
    gmm_results['ari'].append(ari)
    gmm_results['nmi'].append(nmi)
    
    print(f"{n_clusters:<10} {aic:<12.1f} {bic:<12.1f} {log_likelihood:<15.3f} {silhouette:<12.3f} {ari:<8.3f} {nmi:<8.3f}")

# Find optimal number of clusters using different criteria
aic_optimal = n_clusters_range[np.argmin(gmm_results['aic'])]
bic_optimal = n_clusters_range[np.argmin(gmm_results['bic'])]
silhouette_optimal = n_clusters_range[np.argmax(gmm_results['silhouette'])]
ari_optimal = n_clusters_range[np.argmax(gmm_results['ari'])]
nmi_optimal = n_clusters_range[np.argmax(gmm_results['nmi'])]

print(f"\n📈 OPTIMAL CLUSTER ANALYSIS:")
print(f"AIC optimal clusters: {aic_optimal}")
print(f"BIC optimal clusters: {bic_optimal}")
print(f"Silhouette optimal clusters: {silhouette_optimal}")
print(f"ARI optimal clusters: {ari_optimal}")
print(f"NMI optimal clusters: {nmi_optimal}")
print(f"True number of classes: {len(components)}")

# Plot clustering metrics
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# AIC/BIC plot
axes[0, 0].plot(gmm_results['n_clusters'], gmm_results['aic'], 'b-o', label='AIC', linewidth=2, markersize=6)
axes[0, 0].plot(gmm_results['n_clusters'], gmm_results['bic'], 'r-s', label='BIC', linewidth=2, markersize=6)
axes[0, 0].axvline(x=len(components), color='green', linestyle='--', alpha=0.7, label=f'True classes: {len(components)}')
axes[0, 0].set_xlabel('Number of Clusters')
axes[0, 0].set_ylabel('Information Criterion')
axes[0, 0].set_title('AIC/BIC vs Number of Clusters')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Log-likelihood plot
axes[0, 1].plot(gmm_results['n_clusters'], gmm_results['log_likelihood'], 'g-^', linewidth=2, markersize=6)
axes[0, 1].axvline(x=len(components), color='green', linestyle='--', alpha=0.7, label=f'True classes: {len(components)}')
axes[0, 1].set_xlabel('Number of Clusters')
axes[0, 1].set_ylabel('Log-Likelihood')
axes[0, 1].set_title('Log-Likelihood vs Number of Clusters')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Silhouette score plot
axes[0, 2].plot(gmm_results['n_clusters'], gmm_results['silhouette'], 'm-d', linewidth=2, markersize=6)
axes[0, 2].axvline(x=len(components), color='green', linestyle='--', alpha=0.7, label=f'True classes: {len(components)}')
axes[0, 2].set_xlabel('Number of Clusters')
axes[0, 2].set_ylabel('Silhouette Score')
axes[0, 2].set_title('Silhouette Score vs Number of Clusters')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# ARI plot
axes[1, 0].plot(gmm_results['n_clusters'], gmm_results['ari'], 'c-p', linewidth=2, markersize=6)
axes[1, 0].axvline(x=len(components), color='green', linestyle='--', alpha=0.7, label=f'True classes: {len(components)}')
axes[1, 0].set_xlabel('Number of Clusters')
axes[1, 0].set_ylabel('Adjusted Rand Index')
axes[1, 0].set_title('ARI vs Number of Clusters')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# NMI plot
axes[1, 1].plot(gmm_results['n_clusters'], gmm_results['nmi'], 'y-h', linewidth=2, markersize=6)
axes[1, 1].axvline(x=len(components), color='green', linestyle='--', alpha=0.7, label=f'True classes: {len(components)}')
axes[1, 1].set_xlabel('Number of Clusters')
axes[1, 1].set_ylabel('Normalized Mutual Information')
axes[1, 1].set_title('NMI vs Number of Clusters')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Combined metrics plot (normalized)
# Normalize metrics for comparison
aic_norm = 1 - (np.array(gmm_results['aic']) - min(gmm_results['aic'])) / (max(gmm_results['aic']) - min(gmm_results['aic']))
bic_norm = 1 - (np.array(gmm_results['bic']) - min(gmm_results['bic'])) / (max(gmm_results['bic']) - min(gmm_results['bic']))
silhouette_norm = (np.array(gmm_results['silhouette']) - min(gmm_results['silhouette'])) / (max(gmm_results['silhouette']) - min(gmm_results['silhouette']))

axes[1, 2].plot(gmm_results['n_clusters'], aic_norm, 'b-', label='AIC (inverted)', alpha=0.7)
axes[1, 2].plot(gmm_results['n_clusters'], bic_norm, 'r-', label='BIC (inverted)', alpha=0.7)
axes[1, 2].plot(gmm_results['n_clusters'], silhouette_norm, 'm-', label='Silhouette', alpha=0.7)
axes[1, 2].plot(gmm_results['n_clusters'], gmm_results['ari'], 'c-', label='ARI', alpha=0.7)
axes[1, 2].plot(gmm_results['n_clusters'], gmm_results['nmi'], 'y-', label='NMI', alpha=0.7)
axes[1, 2].axvline(x=len(components), color='green', linestyle='--', alpha=0.7, label=f'True classes: {len(components)}')
axes[1, 2].set_xlabel('Number of Clusters')
axes[1, 2].set_ylabel('Normalized Score')
axes[1, 2].set_title('All Metrics (Normalized) vs Number of Clusters')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Step 3: Detailed analysis with optimal cluster number
print("\n🔍 Step 3: Detailed Analysis with Optimal Clustering")
print("-" * 50)

# Use BIC as primary criterion (often more conservative and better for model selection)
optimal_clusters = bic_optimal

print(f"Using {optimal_clusters} clusters (BIC optimal) for detailed analysis...")

# Fit final GMM model
final_gmm = GaussianMixture(n_components=optimal_clusters, random_state=42, covariance_type='full')
final_labels = final_gmm.fit_predict(X_pca)

# Create visualization of clusters in PCA space
if optimal_pca_components >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot clusters
    scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=final_labels, cmap='viridis', alpha=0.6, s=50)
    axes[0].set_xlabel('First Principal Component')
    axes[0].set_ylabel('Second Principal Component')
    axes[0].set_title(f'GMM Clustering Results ({optimal_clusters} clusters)')
    axes[0].grid(True, alpha=0.3)
    plt.colorbar(scatter1, ax=axes[0])
    
    # Plot true labels for comparison
    scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true_numeric, cmap='Set1', alpha=0.6, s=50)
    axes[1].set_xlabel('First Principal Component')
    axes[1].set_ylabel('Second Principal Component')
    axes[1].set_title('True Component Labels')
    axes[1].grid(True, alpha=0.3)
    plt.colorbar(scatter2, ax=axes[1])
    
    plt.tight_layout()
    plt.show()

# Cluster composition analysis
print(f"\n📋 CLUSTER COMPOSITION ANALYSIS:")
print("-" * 40)

# Create confusion matrix between predicted clusters and true labels
cluster_composition = pd.crosstab(final_labels, y_true, margins=True)
print("Cluster vs True Component Composition:")
print(cluster_composition)

# Calculate cluster purity and other metrics
cluster_purities = []
for cluster_id in range(optimal_clusters):
    cluster_mask = final_labels == cluster_id
    if np.sum(cluster_mask) > 0:
        cluster_true_labels = y_true[cluster_mask]
        most_common_class = cluster_true_labels.value_counts().iloc[0]
        purity = most_common_class / len(cluster_true_labels)
        cluster_purities.append(purity)
        
        print(f"\nCluster {cluster_id}:")
        print(f"  Size: {np.sum(cluster_mask)} samples")
        print(f"  Purity: {purity:.3f}")
        print(f"  Composition: {dict(cluster_true_labels.value_counts())}")

overall_purity = np.mean(cluster_purities)
print(f"\nOverall cluster purity: {overall_purity:.3f}")

# Final summary
print(f"\n🎯 FINAL SUMMARY:")
print("=" * 50)
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} wavelengths")
print(f"PCA reduction: {optimal_pca_components} components ({cumulative_variance[optimal_pca_components-2]:.3f} variance)")
print(f"True number of classes: {len(components)}")
print(f"GMM suggested clusters: {optimal_clusters}")
print(f"Clustering agreement with true labels:")
print(f"  • Adjusted Rand Index: {gmm_results['ari'][optimal_clusters-2]:.3f}")
print(f"  • Normalized Mutual Information: {gmm_results['nmi'][optimal_clusters-2]:.3f}")
print(f"  • Silhouette Score: {gmm_results['silhouette'][optimal_clusters-2]:.3f}")
print(f"  • Overall Purity: {overall_purity:.3f}")

if optimal_clusters == len(components):
    print(f"\n✅ SUCCESS: GMM correctly identified {len(components)} classes!")
elif abs(optimal_clusters - len(components)) <= 1:
    print(f"\n⚠️ CLOSE: GMM found {optimal_clusters} vs {len(components)} true classes (difference of {abs(optimal_clusters - len(components))})")
else:
    print(f"\n❌ MISMATCH: GMM found {optimal_clusters} vs {len(components)} true classes (difference of {abs(optimal_clusters - len(components))})")

print(f"\n💡 RECOMMENDATIONS:")
if optimal_clusters == len(components):
    print("  • The dataset has clear class structure that GMM can detect")
    print("  • PCA + GMM is suitable for this classification problem")
elif optimal_clusters > len(components):
    print("  • Data might have sub-clusters within true classes")
    print("  • Consider hierarchical relationships in the data")
else:
    print("  • Some classes might be very similar spectrally")
    print("  • Consider feature engineering or different distance metrics")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("PCA + DBSCAN CLUSTERING ANALYSIS")
print("=" * 80)

# Load and prepare the data
df = pd.read_csv('data/csv/processed_data.csv')

# Identify feature columns (wavelengths)
non_feature_cols = ['tire_number', 'origin', 'measurement_id']
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
wavelength_columns = [col for col in numeric_columns if col not in non_feature_cols]

# Prepare features and true labels
X = df[wavelength_columns]
y_true = df['origin']  # True component labels
components = y_true.unique()
# Step 1: PCA Analysis
print("\n📊 Step 1: PCA Dimensionality Reduction Analysis")
print("-" * 50)

# Find optimal number of PCA components
variance_ratios = []
cumulative_variance = []
n_components_range = range(2, min(101, X.shape[1]))

# Fit PCA with different numbers of components
pca_full = PCA()
pca_full.fit(X)

for n in n_components_range:
    cumulative_var = np.sum(pca_full.explained_variance_ratio_[:n])
    cumulative_variance.append(cumulative_var)

# Find elbow point (95% variance explained)
target_variance = 0.95
optimal_pca_components = None
for i, var in enumerate(cumulative_variance):
    if var >= target_variance:
        optimal_pca_components = n_components_range[i]
        break

if optimal_pca_components is None:
    optimal_pca_components = min(50, X.shape[1] - 1)

print(f"Optimal PCA components for {target_variance*100}% variance: {optimal_pca_components}")
print(f"Variance explained with {optimal_pca_components} components: {cumulative_variance[optimal_pca_components-2]:.3f}")

# Apply PCA with optimal components
pca = PCA(n_components=optimal_pca_components)
X_pca = pca.fit_transform(X)

print(f"Reduced dataset shape: {X_pca.shape}")

# Plot PCA variance analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Cumulative variance plot
ax1.plot(n_components_range, cumulative_variance, 'b-', linewidth=2, marker='o', markersize=4)
ax1.axhline(y=target_variance, color='r', linestyle='--', alpha=0.7, label=f'{target_variance*100}% variance')
ax1.axvline(x=optimal_pca_components, color='r', linestyle='--', alpha=0.7, label=f'Optimal: {optimal_pca_components} components')
ax1.set_xlabel('Number of PCA Components')
ax1.set_ylabel('Cumulative Variance Explained')
ax1.set_title('PCA Cumulative Variance Explained')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Individual variance plot (first 20 components)
individual_variance = pca_full.explained_variance_ratio_[:20]
ax2.bar(range(1, len(individual_variance) + 1), individual_variance, alpha=0.7)
ax2.set_xlabel('PCA Component')
ax2.set_ylabel('Individual Variance Explained')
ax2.set_title('Individual Variance per PCA Component (First 20)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Step 2: Find optimal eps parameter for DBSCAN using k-nearest neighbors
print("\n🎯 Step 2: Finding Optimal DBSCAN Parameters")
print("-" * 50)

# Calculate k-nearest neighbors to find optimal eps
k = min(10, len(components) * 2)  # Use 2x the number of true classes or 10, whichever is smaller
neighbors = NearestNeighbors(n_neighbors=k)
neighbors_fit = neighbors.fit(X_pca)
distances, indices = neighbors_fit.kneighbors(X_pca)

# Sort distances to k-th nearest neighbor
distances = np.sort(distances[:, k-1], axis=0)

# Plot k-distance graph to find elbow (optimal eps)
plt.figure(figsize=(10, 6))
plt.plot(distances, 'b-', linewidth=2)
plt.xlabel('Data Points (sorted by distance)')
plt.ylabel(f'{k}-NN Distance')
plt.title(f'{k}-Distance Graph for DBSCAN eps Parameter Selection')
plt.grid(True, alpha=0.3)

# Find elbow point using gradient method
gradients = np.gradient(distances)
elbow_index = np.argmax(gradients)
optimal_eps = distances[elbow_index]

plt.axhline(y=optimal_eps, color='r', linestyle='--', alpha=0.7, label=f'Suggested eps: {optimal_eps:.3f}')
plt.legend()
plt.show()

print(f"Suggested eps parameter: {optimal_eps:.3f}")

# Step 3: DBSCAN Parameter Testing
print("\n🔍 Step 3: DBSCAN Parameter Grid Search")
print("-" * 50)

# Test different eps values around the optimal
eps_range = np.linspace(optimal_eps * 0.5, optimal_eps * 2.0, 10)
min_samples_range = [3, 5, 8, 10, 15, 20]

dbscan_results = []

print("Testing different DBSCAN parameters...")
print(f"{'Eps':<10} {'MinSamples':<12} {'N_Clusters':<12} {'N_Noise':<10} {'Silhouette':<12} {'ARI':<8} {'NMI':<8}")
print("-" * 80)

# Convert true labels to numeric
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_true_numeric = le.fit_transform(y_true)

best_score = -1
best_params = None
best_labels = None

for eps in eps_range:
    for min_samples in min_samples_range:
        # Fit DBSCAN
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        cluster_labels = dbscan.fit_predict(X_pca)
        
        # Count clusters and noise points
        n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
        n_noise = list(cluster_labels).count(-1)
        
        # Calculate metrics only if we have clusters
        if n_clusters > 1:
            # For silhouette score, exclude noise points (-1 labels)
            if n_noise < len(cluster_labels) - 1:  # Ensure we have enough non-noise points
                mask = cluster_labels != -1
                if len(set(cluster_labels[mask])) > 1:  # Need at least 2 clusters for silhouette
                    silhouette = silhouette_score(X_pca[mask], cluster_labels[mask])
                else:
                    silhouette = -1
            else:
                silhouette = -1
            
            # ARI and NMI can handle noise points
            ari = adjusted_rand_score(y_true_numeric, cluster_labels)
            nmi = normalized_mutual_info_score(y_true_numeric, cluster_labels)
        else:
            silhouette = -1
            ari = 0
            nmi = 0
        
        # Store results
        result = {
            'eps': eps,
            'min_samples': min_samples,
            'n_clusters': n_clusters,
            'n_noise': n_noise,
            'silhouette': silhouette,
            'ari': ari,
            'nmi': nmi,
            'labels': cluster_labels
        }
        dbscan_results.append(result)
        
        # Track best result based on combined score
        combined_score = (ari + nmi) / 2  # Average of ARI and NMI
        if combined_score > best_score and n_clusters > 0:
            best_score = combined_score
            best_params = (eps, min_samples)
            best_labels = cluster_labels
        
        print(f"{eps:<10.3f} {min_samples:<12} {n_clusters:<12} {n_noise:<10} {silhouette:<12.3f} {ari:<8.3f} {nmi:<8.3f}")

print(f"\nBest parameters: eps={best_params[0]:.3f}, min_samples={best_params[1]}")
print(f"Best combined score (ARI+NMI)/2: {best_score:.3f}")

# Step 4: Visualize results with best parameters
print("\n📈 Step 4: Visualization and Analysis")
print("-" * 50)

# Use best parameters for final clustering
best_eps, best_min_samples = best_params
final_dbscan = DBSCAN(eps=best_eps, min_samples=best_min_samples)
final_labels = final_dbscan.fit_predict(X_pca)

n_clusters_final = len(set(final_labels)) - (1 if -1 in final_labels else 0)
n_noise_final = list(final_labels).count(-1)

print(f"Final DBSCAN results:")
print(f"  • Number of clusters: {n_clusters_final}")
print(f"  • Number of noise points: {n_noise_final}")
print(f"  • Noise percentage: {(n_noise_final/len(final_labels)*100):.1f}%")

# Create visualizations
if optimal_pca_components >= 2:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: DBSCAN clusters
    scatter1 = axes[0, 0].scatter(X_pca[:, 0], X_pca[:, 1], c=final_labels, cmap='viridis', alpha=0.6, s=50)
    axes[0, 0].set_xlabel('First Principal Component')
    axes[0, 0].set_ylabel('Second Principal Component')
    axes[0, 0].set_title(f'DBSCAN Clustering Results ({n_clusters_final} clusters + {n_noise_final} noise)')
    axes[0, 0].grid(True, alpha=0.3)
    plt.colorbar(scatter1, ax=axes[0, 0])
    
    # Plot 2: True labels for comparison
    scatter2 = axes[0, 1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true_numeric, cmap='Set1', alpha=0.6, s=50)
    axes[0, 1].set_xlabel('First Principal Component')
    axes[0, 1].set_ylabel('Second Principal Component')
    axes[0, 1].set_title('True Component Labels')
    axes[0, 1].grid(True, alpha=0.3)
    plt.colorbar(scatter2, ax=axes[0, 1])
    
    # Plot 3: Parameter sensitivity heatmap (ARI)
    # Create parameter grid for heatmap
    eps_grid = sorted(list(set([r['eps'] for r in dbscan_results])))
    min_samples_grid = sorted(list(set([r['min_samples'] for r in dbscan_results])))
    
    ari_matrix = np.zeros((len(min_samples_grid), len(eps_grid)))
    for i, ms in enumerate(min_samples_grid):
        for j, eps in enumerate(eps_grid):
            result = next((r for r in dbscan_results if r['eps'] == eps and r['min_samples'] == ms), None)
            if result:
                ari_matrix[i, j] = result['ari']
    
    im1 = axes[1, 0].imshow(ari_matrix, cmap='viridis', aspect='auto')
    axes[1, 0].set_xlabel('Eps Parameter')
    axes[1, 0].set_ylabel('Min Samples Parameter')
    axes[1, 0].set_title('ARI Score Heatmap')
    axes[1, 0].set_xticks(range(len(eps_grid)))
    axes[1, 0].set_xticklabels([f'{eps:.2f}' for eps in eps_grid], rotation=45)
    axes[1, 0].set_yticks(range(len(min_samples_grid)))
    axes[1, 0].set_yticklabels(min_samples_grid)
    plt.colorbar(im1, ax=axes[1, 0])
    
    # Plot 4: Number of clusters heatmap
    clusters_matrix = np.zeros((len(min_samples_grid), len(eps_grid)))
    for i, ms in enumerate(min_samples_grid):
        for j, eps in enumerate(eps_grid):
            result = next((r for r in dbscan_results if r['eps'] == eps and r['min_samples'] == ms), None)
            if result:
                clusters_matrix[i, j] = result['n_clusters']
    
    im2 = axes[1, 1].imshow(clusters_matrix, cmap='plasma', aspect='auto')
    axes[1, 1].set_xlabel('Eps Parameter')
    axes[1, 1].set_ylabel('Min Samples Parameter')
    axes[1, 1].set_title('Number of Clusters Heatmap')
    axes[1, 1].set_xticks(range(len(eps_grid)))
    axes[1, 1].set_xticklabels([f'{eps:.2f}' for eps in eps_grid], rotation=45)
    axes[1, 1].set_yticks(range(len(min_samples_grid)))
    axes[1, 1].set_yticklabels(min_samples_grid)
    plt.colorbar(im2, ax=axes[1, 1])
    
    plt.tight_layout()
    plt.show()

# Step 5: Detailed cluster analysis
print("\n📋 Step 5: Detailed Cluster Analysis")
print("-" * 50)

# Cluster composition analysis
unique_clusters = sorted([c for c in set(final_labels) if c != -1])
noise_mask = final_labels == -1

if len(unique_clusters) > 0:
    print("Cluster vs True Component Composition:")
    
    # Create a modified labels array for crosstab (handle noise separately)
    labels_for_crosstab = final_labels.copy()
    
    if -1 in final_labels:
        # Create crosstab including noise as a separate category
        cluster_composition = pd.crosstab(
            pd.Series(labels_for_crosstab, name='DBSCAN_Cluster'), 
            pd.Series(y_true, name='True_Component'), 
            margins=True
        )
    else:
        cluster_composition = pd.crosstab(final_labels, y_true, margins=True)
    
    print(cluster_composition)
    
    # Calculate cluster purity (excluding noise)
    cluster_purities = []
    for cluster_id in unique_clusters:
        cluster_mask = final_labels == cluster_id
        if np.sum(cluster_mask) > 0:
            cluster_true_labels = y_true[cluster_mask]
            most_common_class = cluster_true_labels.value_counts().iloc[0]
            purity = most_common_class / len(cluster_true_labels)
            cluster_purities.append(purity)
            
            print(f"\nCluster {cluster_id}:")
            print(f"  Size: {np.sum(cluster_mask)} samples")
            print(f"  Purity: {purity:.3f}")
            print(f"  Composition: {dict(cluster_true_labels.value_counts())}")
    
    if n_noise_final > 0:
        noise_true_labels = y_true[noise_mask]
        print(f"\nNoise Points ({n_noise_final} samples):")
        print(f"  Composition: {dict(noise_true_labels.value_counts())}")
    
    overall_purity = np.mean(cluster_purities) if cluster_purities else 0
    print(f"\nOverall cluster purity (excluding noise): {overall_purity:.3f}")

# Calculate final metrics
final_ari = adjusted_rand_score(y_true_numeric, final_labels)
final_nmi = normalized_mutual_info_score(y_true_numeric, final_labels)

if n_clusters_final > 1 and n_noise_final < len(final_labels) - 1:
    mask = final_labels != -1
    if len(set(final_labels[mask])) > 1:
        final_silhouette = silhouette_score(X_pca[mask], final_labels[mask])
    else:
        final_silhouette = -1
else:
    final_silhouette = -1

# Final summary
print(f"\n🎯 FINAL SUMMARY:")
print("=" * 50)
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} wavelengths")
print(f"PCA reduction: {optimal_pca_components} components ({cumulative_variance[optimal_pca_components-2]:.3f} variance)")
print(f"True number of classes: {len(components)}")
print(f"DBSCAN found clusters: {n_clusters_final}")
print(f"Noise points: {n_noise_final} ({(n_noise_final/len(final_labels)*100):.1f}%)")
print(f"Best parameters: eps={best_eps:.3f}, min_samples={best_min_samples}")
print(f"Clustering agreement with true labels:")
print(f"  • Adjusted Rand Index: {final_ari:.3f}")
print(f"  • Normalized Mutual Information: {final_nmi:.3f}")
print(f"  • Silhouette Score: {final_silhouette:.3f}")
if cluster_purities:
    print(f"  • Overall Purity: {overall_purity:.3f}")

# Provide interpretation
if n_clusters_final == len(components):
    print(f"\n✅ SUCCESS: DBSCAN correctly identified {len(components)} classes!")
elif abs(n_clusters_final - len(components)) <= 1:
    print(f"\n⚠️ CLOSE: DBSCAN found {n_clusters_final} vs {len(components)} true classes (difference of {abs(n_clusters_final - len(components))})")
else:
    print(f"\n❌ MISMATCH: DBSCAN found {n_clusters_final} vs {len(components)} true classes (difference of {abs(n_clusters_final - len(components))})")

print(f"\n💡 DBSCAN INSIGHTS:")
if n_noise_final > 0:
    print(f"  • {(n_noise_final/len(final_labels)*100):.1f}% of data points are classified as noise/outliers")
    print(f"  • This suggests some data points don't fit well into any cluster")

if final_ari > 0.7:
    print("  • High agreement with true labels - DBSCAN is effective for this data")
elif final_ari > 0.4:
    print("  • Moderate agreement with true labels - some cluster structure detected")
else:
    print("  • Low agreement with true labels - data may not have clear density-based clusters")

print(f"\n💡 RECOMMENDATIONS:")
if n_clusters_final == len(components) and final_ari > 0.7:
    print("  • DBSCAN successfully identified the class structure")
    print("  • Consider using density-based approaches for classification")
elif n_noise_final > len(final_labels) * 0.1:
    print("  • High noise percentage suggests trying different eps/min_samples values")
    print("  • Consider preprocessing or outlier detection")
else:
    print("  • DBSCAN may not be optimal for this data structure")
    print("  • Consider other clustering algorithms or feature engineering")